# نوفا الصغير — المرحلة الثانية: إضافة الصورة والصوت (Vision + Audio)

هذا الدفتر **يكمل** تدريب النموذج الذي أنتجه دفتر المرحلة الأولى (النص) — لا يبدأ من الصفر، ولا يفقد أي شيء تعلّمه النموذج من النص. يضيف عليه فقط قدرتين حقيقيتين جديدتين:
- **الصورة**: توليد صور من نص (Generation) + وصف/فهم صورة معطاة (Understanding) — بالاتجاهين معاً من نفس البيانات.
- **الصوت**: تحويل نص لصوت (Generation) + تفريغ/فهم صوت معطى (Understanding) — بالاتجاهين معاً أيضاً.

**نطاق حقيقي يجب معرفته بصراحة:** بيانات الصور المتاحة بكثرة وجاهزة للسحب الآن هي بتعليقات **إنجليزية** (Flickr30k) — لا يوجد مصدر عربي مكافئ بنفس الحجم متاح فوراً. هذا لا يضر الآلية نفسها (تعلّم ربط البكسل برموز، وربطها بنص التعليق) لأنها مستقلة عن اللغة تماماً، لكن يعني أن وصف الصور بالعربية الفصيحة يحتاج بيانات عربية لاحقاً لتحسينه أكثر. الصوت بالمقابل عربي حقيقي بالكامل (Common Voice العربية).

## قبل الضغط على "Save Version → Save & Run All" — 3 خطوات:

1. **أضف نتاج (Output) دفتر المرحلة الأولى كمدخل هنا:** من القائمة الجانبية اضغط **+ Add Input → Notebook Output**، واختر دفتر `sham_small_training` (مرحلة النص) الذي انتهى تدريبه. هذا يجعل نقطة الحفظ وأداة تقسيم النص من المرحلة الأولى متاحتين هنا.
2. **نفس أسرار Kaggle المستخدمة سابقاً:** تأكد أن `GITHUB_TOKEN` لا يزال مضافاً في Add-ons → Secrets (لا حاجة لإضافته من جديد إن كان موجوداً من قبل).
3. **فعّل الإنترنت واختر GPU** من Settings، تماماً كما في المرحلة الأولى.

**مهم:** استخدم **Save Version → Save & Run All (Commit)** مباشرة من البداية لهذا الدفتر (وليس التشغيل التفاعلي) — تعلمنا من المرحلة الأولى أن هذا هو الأسلوب الآمن الوحيد لتشغيل يستغرق ساعات دون قلق من إغلاق الهاتف أو انقطاع الاتصال.


### 1) سحب الكود الحقيقي من GitHub مباشرة (نفس أسلوب المرحلة الأولى)

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 3) تحميل نقطة الحفظ وأداة تقسيم النص من المرحلة الأولى

هذه الخلية تبحث عن نتاج دفتر المرحلة الأولى الذي أضفته كـ Input (الخطوة 1 أعلاه) — إن لم تجدها، ستتوقف برسالة واضحة تخبرك بالضبط ما الناقص.


In [ ]:
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — فعّله من Settings → Accelerator)" if device == "cpu" else ""))

stage1_checkpoints = sorted(
    Path("/kaggle/input").rglob("*.pt"),  # يعمل بأي ترتيب مجلدات — لا يشترط اسم "checkpoints" تحديداً
    key=lambda p: p.stat().st_mtime,
)
assert stage1_checkpoints, (
    "لم يتم العثور على أي نقطة حفظ من المرحلة الأولى تحت /kaggle/input — "
    "تأكد أنك أضفت نتاج دفتر المرحلة الأولى (sham_small_training) كـ Input لهذا الدفتر (راجع الخطوة 1 أعلاه)."
)
stage1_checkpoint_path = stage1_checkpoints[-1]
model, start_step, _ = load_checkpoint(stage1_checkpoint_path, map_location=device)
print(f"تم تحميل نقطة حفظ حقيقية من المرحلة الأولى: {stage1_checkpoint_path} (الخطوة {start_step:,})")

tokenizer_candidates = list(Path("/kaggle/input").rglob("sham_small_tokenizer.json"))
assert tokenizer_candidates, (
    "لم يتم العثور على sham_small_tokenizer.json من المرحلة الأولى — تأكد من إضافة نتاج دفتر المرحلة الأولى كـ Input."
)
tokenizer = ShamTextTokenizer.load(tokenizer_candidates[0])
print(f"تم تحميل أداة تقسيم النص الحقيقية من المرحلة الأولى (vocab={tokenizer.vocab_size}).")


### 4) جمع بيانات صورة وصوت حقيقية

- **الصور**: نحاول أولاً مصدراً عربياً حقيقياً حديثاً (`Misraj/Arabic-Image-Captioning_100M` — 100 مليون زوج صورة+تعليق عربي حقيقي)، والخلية أدناه تتراجع تلقائياً لـ Flickr30k (تعليقات إنجليزية) إن فشل المصدر العربي لأي سبب واقعي (تغيّر اسم عمود، تغيّر الرابط، إلخ) — **هذا تحقّق حقيقي وقت التشغيل، وليس افتراضاً**: لم أتمكن من فتح صفحة هذا المصدر مباشرة للتأكد من أسماء أعمدته بالضبط (huggingface.co محجوب من بيئة التطوير التي بنيت منها هذا الكود)، فالكود يفحص الأعمدة الحقيقية فور وصول أول عنصر حقيقي، ويخبرك فوراً إن كانت مختلفة عمّا توقعته بدل الفشل الصامت.
- **الصوت**: Common Voice العربية (صوت وتفريغ نصي عربي حقيقي وموثوق)، عبر streaming.

ابدأ بعدد معقول (كما في المرحلة الأولى) للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق.


In [ ]:
from sham_data_sources import Ledger, collect

MAX_IMAGES = 3_000
MAX_AUDIO_SAMPLES = 3_000

# كل تشغيل يأخذ بيانات جديدة لم يرها النموذج من قبل: موضع استئناف لكل مصدر + بصمة لكل
# عيّنة، في سجلّين خاصين بالمرحلة الثانية (stage2_image / stage2_audio) يُحفظان في
# /kaggle/working/checkpoints — أضف نتاج هذا الدفتر كمدخل للتشغيل التالي ليكمل بعده.
# (سابقاً: نفس أول 3000 صورة في كل تشغيل، ومصدر Common Voice صار فارغاً على Hugging Face.)
image_ledger = Ledger.load("image", name="stage2_image")
audio_ledger = Ledger.load("audio", name="stage2_audio")
image_manifest, image_collect_stats = collect("image", "/kaggle/working/corpus/images", MAX_IMAGES, image_ledger, image_size=64)
audio_manifest, audio_collect_stats = collect("audio", "/kaggle/working/corpus/audio", MAX_AUDIO_SAMPLES, audio_ledger)
Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
image_ledger.save("/kaggle/working/checkpoints")
audio_ledger.save("/kaggle/working/checkpoints")
assert image_collect_stats.written + audio_collect_stats.written > 0, (
    "لا توجد بيانات جديدة غير مكررة في أي مصدر — المرحلة الثانية رأت كل البيانات المتاحة؛ أضف مصادر جديدة إلى SOURCES."
)
print(f"جاهز: {image_manifest}")
print(f"جاهز: {audio_manifest}")


### 5) تدريب أداتي ترميز الصورة والصوت (VQ-VAE) على البيانات الحقيقية

هاتان الأداتان تحوّلان صورة/مقطع صوت حقيقياً إلى "رموز" (tokens) يفهمها النموذج الرئيسي — يجب تدريبهما أولاً على بيانات حقيقية قبل استخدامهما (تدريبهما بأوزان عشوائية ينتج ضجيجاً بلا معنى).

حجم أداة ترميز الصورة هنا هو حجم "بداية" آمن (صور 64×64) يتناسب مع GPU المجاني — قابل للتكبير لاحقاً بنفس فلسفة النموذج الرئيسي.


In [ ]:
import torch
from PIL import Image
import json as _json
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig
from train_image_tokenizer import train_vqvae as train_image_vqvae, save_tokenizer_checkpoint as save_image_tokenizer
from train_image_tokenizer import load_tokenizer_checkpoint as load_image_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصورة مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي image_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (image):")
_picked_image = select_pretrained_tokenizer("image")
_pretrained_image = [_picked_image[2]] if _picked_image else []
if _picked_image:
    image_tokenizer, _image_step = _picked_image[0], _picked_image[1]
    image_tokenizer_cfg = image_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصورة المدرّبة مسبقاً: {_pretrained_image[-1]} (خطوة {_image_step})")
    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=_image_step)
else:
    print("لم يُعثر على أداة ترميز الصورة مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")


    image_tokenizer_cfg = ImageTokenizerConfig(
        image_size=64, base_channels=32, channel_multipliers=(1, 2, 4), code_dim=64,  # حجم بداية آمن
        # num_codes يبقى ثابتاً على القيمة المعمارية الحقيقية (8192) — لا يُغيَّر أبداً، فالنموذج الرئيسي يحجز لها هذا المدى بالضبط.
    )
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    print(f"عدد رموز كل صورة: {image_tokenizer_cfg.tokens_per_image} (شبكة {image_tokenizer_cfg.latent_grid_size}x{image_tokenizer_cfg.latent_grid_size})")

    # تحميل الصور الحقيقية إلى مصفوفة واحدة في الذاكرة (مناسب لبضعة آلاف صورة بحجم 64x64)
    image_root = Path(image_manifest).parent
    images_list = []
    with open(image_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            img = Image.open(image_root / record["image"]).convert("RGB").resize((image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size))
            tensor = torch.tensor(list(img.getdata()), dtype=torch.float32).view(image_tokenizer_cfg.image_size, image_tokenizer_cfg.image_size, 3)
            images_list.append(tensor.permute(2, 0, 1) / 127.5 - 1.0)
    real_images = torch.stack(images_list, dim=0)
    print(f"عدد الصور الحقيقية المحمّلة للتدريب: {real_images.shape[0]:,}")

    image_vqvae_stats = train_image_vqvae(image_tokenizer, real_images, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {image_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {image_vqvae_stats.final_codebook_usage}/{image_vqvae_stats.codebook_size}")

    save_image_tokenizer("/kaggle/working/image_tokenizer.pt", image_tokenizer, step=len(image_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصورة الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
image_tokenizer = image_tokenizer.to("cpu").eval()


In [ ]:
import soundfile as sf
from mel_spectrogram import waveform_to_mel_spectrogram
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
from train_audio_tokenizer import train_vqvae as train_audio_vqvae, save_tokenizer_checkpoint as save_audio_tokenizer
from train_audio_tokenizer import load_tokenizer_checkpoint as load_audio_tokenizer

# ربط تلقائي بشام: إن كانت أداة ترميز الصوت مدرّبة مسبقاً في مسارها المستقل (CPU)
# ومُضافة كمُدخل (Add Input) لهذا الدفتر، نستخدمها كما هي بدل تدريب أداة
# جديدة صغيرة من الصفر — لا حاجة لمعرفة اسم المجموعة أو مسارها، البحث تلقائي.
# يقبل المسارين تلقائياً بلا تعارض: أي مجموعة مُضافة تحوي audio_tokenizer.pt
# (مسار الترميز المستقل، أو مجموعة مدمجة) — يُختار الأكثر تدريباً والمتوافق
# مع النموذج (tokenizer_select.py)، وإن لم يوجد شيء صالح يُدرَّب هنا من جديد.
from tokenizer_select import select_pretrained_tokenizer
print("البحث عن أداة ترميز جاهزة (audio):")
_picked_audio = select_pretrained_tokenizer("audio")
_pretrained_audio = [_picked_audio[2]] if _picked_audio else []
if _picked_audio:
    audio_tokenizer, _audio_step = _picked_audio[0], _picked_audio[1]
    audio_tokenizer_cfg = audio_tokenizer.cfg
    print(f"✅ استُخدمت أداة ترميز الصوت المدرّبة مسبقاً: {_pretrained_audio[-1]} (خطوة {_audio_step})")
    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=_audio_step)
else:
    print("لم يُعثر على أداة ترميز الصوت مدرّبة ضمن المدخلات — تدريب أداة جديدة هنا.")

    audio_tokenizer_cfg = AudioTokenizerConfig()  # الحجم الافتراضي مناسب هنا (num_codes يطابق AUDIO_VOCAB_SIZE أصلاً)
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    print(f"عدد رموز كل مقطع صوتي: {audio_tokenizer_cfg.tokens_per_segment}")

    audio_root = Path(audio_manifest).parent
    mels_list = []
    with open(audio_manifest, encoding="utf-8") as f:
        for line in f:
            record = _json.loads(line)
            waveform, sample_rate = sf.read(str(audio_root / record["audio"]), dtype="float32", always_2d=False)
            if waveform.ndim > 1:
                waveform = waveform.mean(axis=1)
            mel = waveform_to_mel_spectrogram(
                torch.from_numpy(waveform), sample_rate, audio_tokenizer_cfg.n_mels, audio_tokenizer_cfg.segment_frames
            )
            mels_list.append(mel)
    real_mels = torch.stack(mels_list, dim=0)
    print(f"عدد المقاطع الصوتية الحقيقية المحمّلة للتدريب: {real_mels.shape[0]:,}")

    audio_vqvae_stats = train_audio_vqvae(audio_tokenizer, real_mels, num_epochs=10, batch_size=32, device=device, log_every=2)
    print(f"آخر خسارة إعادة بناء+VQ حقيقية: {audio_vqvae_stats.epoch_losses[-1]:.4f} | "
          f"استخدام القاموس (codebook): {audio_vqvae_stats.final_codebook_usage}/{audio_vqvae_stats.codebook_size}")

    save_audio_tokenizer("/kaggle/working/audio_tokenizer.pt", audio_tokenizer, step=len(audio_vqvae_stats.epoch_losses))
    print("تم حفظ أداة ترميز الصوت الحقيقية المدرَّبة.")

# الترميز داخل MultimodalCollator يجري على CPU (الصور/المقاطع تُحمَّل على CPU)،
# ثم تُنقل الدفعات الجاهزة إلى GPU للنموذج الرئيسي — لذا تبقى أداة الترميز على
# CPU وفي وضع التقييم (مجمّدة) في الحالتين: المدرّبة مسبقاً والمدرّبة هنا.
audio_tokenizer = audio_tokenizer.to("cpu").eval()


### 6) بناء دفعات تدريب حقيقية متعددة الوسائط (نص + صورة + صوت معاً)

كل دفعة هنا زوج حقيقي (input_ids, labels) — وليس نصاً عادياً — لأن الصور/الصوت تحتاج حشواً (padding)، تماماً كما تحقق ذلك في الإصلاح الذي أُجري على `train.py` مسبقاً.


In [ ]:
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

image_dataset = ImageCaptionDataset(image_manifest, image_size=image_tokenizer_cfg.image_size)
audio_dataset = AudioTranscriptDataset(audio_manifest, n_mels=audio_tokenizer_cfg.n_mels, segment_frames=audio_tokenizer_cfg.segment_frames)
print(f"عدد أزواج (صورة، تعليق) الآمنة: {len(image_dataset)} ({image_dataset.skipped_entries} استُبعد لأسباب أمان)")
print(f"عدد أزواج (صوت، نص) الآمنة: {len(audio_dataset)} ({audio_dataset.skipped_entries} استُبعد لأسباب أمان)")

image_collator = MultimodalCollator(tokenizer, image_tokenizer, both_directions=True)
audio_collator = AudioMultimodalCollator(tokenizer, audio_tokenizer, both_directions=True)

IMAGE_BATCH_SIZE = 4
AUDIO_BATCH_SIZE = 4

multimodal_batches = []
for start in range(0, len(image_dataset) - (len(image_dataset) % IMAGE_BATCH_SIZE), IMAGE_BATCH_SIZE):
    batch = [image_dataset[i] for i in range(start, start + IMAGE_BATCH_SIZE)]
    multimodal_batches.append(image_collator(batch))
for start in range(0, len(audio_dataset) - (len(audio_dataset) % AUDIO_BATCH_SIZE), AUDIO_BATCH_SIZE):
    batch = [audio_dataset[i] for i in range(start, start + AUDIO_BATCH_SIZE)]
    multimodal_batches.append(audio_collator(batch))

import random
random.Random(0).shuffle(multimodal_batches)  # لا نريد كل الصور أولاً ثم كل الصوت — نخلطهما فعلياً
print(f"عدد دفعات (نص+صورة) و(نص+صوت) الحقيقية الجاهزة: {len(multimodal_batches):,} "
      f"(كل صورة/مقطع يُدرَّب بالاتجاهين معاً: توليد + فهم).")

assert multimodal_batches, "لا توجد دفعات كافية — كبّر MAX_IMAGES/MAX_AUDIO_SAMPLES في خلية جمع البيانات."


### 7) قياس السرعة الحقيقية ثم التدريب الحقيقي

نفس مبدأ المرحلة الأولى بالضبط: نقيس فعلياً بدل التخمين، ثم ندرّب لعدد خطوات واقعي يتناسب مع جلسة 9 ساعات.


In [ ]:
import time
from train import TrainConfig, build_optimizer, train

model.to(device)
CALIBRATION_BATCHES = min(10, len(multimodal_batches))
_calib_optimizer = build_optimizer(model, lr=1e-4, weight_decay=0.1)

model.train()
t0 = time.time()
for batch in multimodal_batches[:CALIBRATION_BATCHES]:
    input_ids, labels = batch[0].to(device), batch[1].to(device)
    _, loss = model(input_ids, labels=labels)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
elapsed = time.time() - t0
steps_per_second = CALIBRATION_BATCHES / elapsed

# غيّر هذا حسب رصيد ساعات GPU المتبقي لك فعلياً في Kaggle (وليس حسب حد
# Kaggle الأقصى نفسه) — بهامش أمان تحته.
MAX_TRAINING_HOURS = 3.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)
print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي ضمن {MAX_TRAINING_HOURS} ساعة: {realistic_steps_for_session:,}")

TOTAL_STEPS = max(min(realistic_steps_for_session, len(multimodal_batches) * 20), 200)
if len(multimodal_batches) < TOTAL_STEPS:
    repeats = (TOTAL_STEPS // len(multimodal_batches)) + 1
    training_batches = (multimodal_batches * repeats)[:TOTAL_STEPS]
    print(f"تم تكرار بيانات الصورة/الصوت {repeats} مرة/مرات للوصول إلى {TOTAL_STEPS:,} خطوة.")
else:
    training_batches = multimodal_batches[:TOTAL_STEPS]

train_cfg = TrainConfig(
    seq_len=model.cfg.max_seq_len, batch_size=IMAGE_BATCH_SIZE, grad_accum_steps=4, lr=1e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=100, log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)
loss_history = train(model, training_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=_calib_optimizer)
print(f"\nانتهى تدريب المرحلة الثانية على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")


### 8) حفظ النتيجة النهائية

النموذج الآن يحمل قدرة النص من المرحلة الأولى + قدرة الصورة والصوت (توليداً وفهماً) من هذه المرحلة، في نفس الأوزان. اضغط **Save Version** بعد انتهاء هذه الخلية ليُحفظ كل شيء كـ Output دائم.


In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final_multimodal.pt", model, final_step)
print(f"تم حفظ النموذج النهائي (نص+صورة+صوت) عند الخطوة {final_step:,}.")
print("أداتا ترميز الصورة والصوت المدرَّبتان محفوظتان أيضاً في /kaggle/working/ (image_tokenizer.pt, audio_tokenizer.pt).")
print("\nاضغط Save Version الآن لحفظ كل هذا كـ Output دائم قابل لإضافته كـ Input لخدمة serve.py أو لتدريب لاحق.")
